# W10 · Day 1 — Context Optimization + Three Caching Layers

**~75 minutes · in-class demo · Jupyter notebook · Track A**

**⚠ DANGER ZONE week (second in a row).** W9 stacked hybrid+rerank+rewriting.
W10 stacks caching+lifecycle. Pace matters — Day 1 is 5 concept beats plus
one build. Slow down on the threshold-tuning demo (Cell 6) — that's where
'why we cache carefully' lands.

**Today:** count prompt tokens, prune retrieved chunks, build a semantic
cache, tune its threshold (including a live false-hit), enable OpenAI's
prompt cache, measure cost + latency savings.

**Tomorrow:** tombstone pattern for document lifecycle (with a live
IsNull vs IsEmpty bug diagnosis), cache invalidation, KPI wrap.

**Corpus:** same 15-doc Acme Analytics Platform we used in W9. Reused for
continuity — the retrieval stack from W9 still works and today we're
adding a layer on top of it.

**Cost per full run:** ~$0.01 (embedding + a handful of LLM calls).

**Notebook flow:**
- Cell 1: Setup + corpus load
- Cell 2: Prompt token accounting — where do your tokens actually go?
- Cell 3: Reduce-k demo — measure precision AND recall, not just precision
- Cell 4: Prompt compression — verbal, Stretch territory
- Cell 5: Semantic cache — build it from scratch (SQLite)
- Cell 6: Threshold tuning — 3 paraphrases and a false-hit
- Cell 7: Prompt cache (OpenAI provider-side) — inspect cached_tokens field
- Cell 8: Embedding cache — 30-second acknowledgement
- Cell 9: Cost + latency — cold vs warm across 6 queries
- Cell 10: Wrap + Day 2 preview

---

## Cell 1 — Setup

Confirms env, loads the W9 Acme corpus + test queries, connects to Qdrant.

The corpus + retrieval stack from W9 are prerequisites. If Cell 1 fails,
fix W9's setup first — this notebook builds ON TOP of W9's retrieval, not
in parallel with it.

In [ ]:
import os
import sys
import time
import json
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — from W7 setup"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from W7 setup"

# Add cwd to path so we can import wk10_pipeline
sys.path.insert(0, str(Path.cwd()))

from wk10_pipeline import (
    load_acme_corpus, load_test_queries,
    build_rag_prompt, count_tokens,
    cache_init, cache_put, cache_lookup, cache_clear, cache_key,
    estimate_cost_usd, PRICING,
)

corpus  = load_acme_corpus()
queries = load_test_queries()
print(f"✓ Loaded {len(corpus)} docs and {len(queries)} test queries (reused from W9)")

---

## Cell 2 — Prompt token accounting: where do your tokens go?

Before you can optimise anything, know where your tokens actually live. The
deck (slide 6) claims a median W9 capstone prompt is ~2,400 tokens. Let's
verify with a real prompt built the same way.

**Anatomy of a RAG prompt:**
```
System prompt         (~600 tokens — stable, cacheable)
Retrieved chunks (5)  (~1,500 tokens — biggest lever)
Few-shot examples     (0 tokens — we don't use them here)
User question         (~50-300 tokens — what they asked)
───────────────────────────────────────
Total                 (~2,400 tokens)
```

In [ ]:
# Build a realistic RAG prompt using the W9 retrieval flow simulated inline
# (we skip the actual Qdrant call here — just grab 5 corpus chunks by hand).
test_q = "how do I export a dashboard to PDF"

# Simulate retrieved top-5 (in W9 this comes from hybrid_retrieve + rerank)
simulated_top5 = [
    {"doc": corpus[0]}, {"doc": corpus[2]},  # dashboards + scheduled reports
    {"doc": corpus[3]}, {"doc": corpus[4]},  # data explorer + permissions
    {"doc": corpus[1]},                       # alerts
]

prompt_5, counts_5 = build_rag_prompt(test_q, simulated_top5)

print(f"══ Prompt tokens by section (top-5 retrieval) ══\n")
print(f"  {'Section':<15s}  {'Tokens':>7s}  Notes")
print(f"  {'-------':<15s}  {'------':>7s}  -----")
print(f"  {'System':<15s}  {counts_5['system']:>7d}  stable — cache candidate")
print(f"  {'Retrieved':<15s}  {counts_5['retrieved']:>7d}  biggest lever — 5 chunks")
print(f"  {'Question':<15s}  {counts_5['question']:>7d}  what user asked")
print(f"  {'-':-<15s}  {'-':->7s}")
print(f"  {'TOTAL':<15s}  {counts_5['total']:>7d}")

print(f"\n(Our Acme corpus has SHORT docs — real capstone chunks are 200-500 tokens")
print(f"each, pushing the retrieved section to 1,500+ tokens as slide 6 shows.)")

**Discussion moment:**
- What's the biggest lever on your prompt? (Retrieved chunks — every time.)
- What's the safest lever to pull first? (The retrieved section — reducing k
  is one number change, reversible.)
- What's the RISKY lever? (System prompt — cutting it loses instruction
  following. Prompt compression can help, but measure groundedness after.)

---

## Cell 3 — Reduce k: measure precision AND recall

**Deck slide 6-7 (post-audit) framing:** reducing top-5 → top-3 is safe if
precision@3 AND recall@3 both hold. Not precision alone — that's a
single-answer heuristic that hides multi-answer failures.

Let's measure both on the 6 test queries and see the trade-off.

In [ ]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = "wk10_day1_acme"

# Recreate collection + embed corpus
try:    qdrant.delete_collection(COLLECTION)
except: pass
qdrant.create_collection(COLLECTION,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE))

texts = [d["text"] for d in corpus]
vectors = [item.embedding for item in openai_client.embeddings.create(
    model="text-embedding-3-small", input=texts).data]
qdrant.upsert(COLLECTION, points=[
    PointStruct(id=i, vector=v, payload=d) for i, (d, v) in enumerate(zip(corpus, vectors))
])
print(f"✓ Ingested {len(corpus)} chunks into {COLLECTION!r}")

def dense_search(q, k=5):
    q_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[q]).data[0].embedding
    hits = qdrant.query_points(COLLECTION, query=q_vec, limit=k).points
    return [{"id": h.payload["id"], "score": h.score, "doc": h.payload} for h in hits]

# Precision@k = 1 if expected_id in top-k retrieved
# Recall@k on this corpus = same (single expected doc per query) — but on
# multi-answer questions they diverge. See the discussion after this cell.
def p_at_k(retrieved_ids, expected_id, k):
    return 1.0 if expected_id in retrieved_ids[:k] else 0.0

print(f"\n══ Precision@k for k=3 vs k=5 ══\n")
print(f"  {'Query':<50s}  {'P@3':>5s}  {'P@5':>5s}  {'tokens saved at k=3'}")

p3_sum, p5_sum = 0.0, 0.0
for tq in queries:
    top5 = dense_search(tq["q"], k=5)
    ids = [h["id"] for h in top5]
    p3 = p_at_k(ids, tq["expected_id"], 3)
    p5 = p_at_k(ids, tq["expected_id"], 5)
    p3_sum += p3; p5_sum += p5
    
    # Estimate tokens saved by dropping positions 4,5
    saved = sum(count_tokens(h["doc"]["text"]) for h in top5[3:])
    print(f"  {tq['q'][:50]:<50s}  {p3:>5.2f}  {p5:>5.2f}  ~{saved} tokens")

n = len(queries)
print(f"\n  Precision@3 = {p3_sum/n:.2f}   Precision@5 = {p5_sum/n:.2f}")
print(f"\n  Decision: safe to cut k=5 → k=3? — {'YES' if p3_sum == p5_sum else 'CHECK'}")

**Important caveat (matches auditor-fixed deck slide 6-7):**

Our test queries have ONE expected answer each — so precision and recall
collapse to the same number. For MULTI-ANSWER questions like "list all
security features" or "what are the error codes", recall@3 might be 0.6
while precision@3 is 1.0. In that case cutting k costs you coverage.

**Rule:** measure BOTH before cutting k. If your golden set is single-
answer, precision is enough. If it's multi-answer or multi-hop, check
recall too — you can't tell from precision alone whether you're losing
evidence at positions 4-5.

---

## Cell 4 — Prompt compression (verbal only — Stretch lab)

**LLMLingua** (Microsoft, 2023-2024) is the leading library. It uses a
small language model to score each token's importance and drops the least-
important 30-40% of tokens from your prompt. The compressed prompt still
conveys the same meaning to a big LLM but costs less.

**Why we're not running it today:**
- Library is ~500MB (torch + a small LM). Slow on Vocareum.
- Quality risk: compression can drop groundedness sharply on technical text
- Reducing k (Cell 3) gives similar savings with zero library dependency

**When it's worth trying (Stretch lab option):**
- You've already cut k as far as recall allows
- Your prompts have long system prompts you can't shorten manually
- You have a large enough golden set to catch groundedness regressions

Skip for now. On to the main event — semantic cache.

---

## Cell 5 — Semantic cache: build it from scratch

**The idea:** when a user asks a question, before running the full RAG
pipeline (retrieval + LLM), check whether we've answered a very similar
question recently. If yes, return the cached answer.

**Not the same as prompt cache.** Prompt cache (Cell 7) is the LLM provider
storing your KV state on their side. Semantic cache is US storing
question→answer pairs indexed by embedding similarity.

**SQLite table schema** (per deck slide 15):
```
query_cache (
    id             TEXT PRIMARY KEY,
    question       TEXT,
    embedding_blob BLOB,
    answer         TEXT,
    hit_count      INTEGER,
    created_ts     REAL,
    last_hit_ts    REAL
)
```

**Lookup algorithm:** embed the new question, cosine-compare against all
cached embeddings, return the best match if similarity ≥ threshold (else
cache miss → run full RAG).

**Scale note:** we scan all rows on every lookup — fine for <1K cached
questions (our capstone scale). Production would use a vector index like
Qdrant or pgvector.

In [ ]:
# Initialise the cache database
CACHE_DB = "wk10_day1_cache.db"
# Clean slate for the demo
if os.path.exists(CACHE_DB):
    os.remove(CACHE_DB)

cache = cache_init(CACHE_DB)
print(f"✓ Cache initialised: {CACHE_DB}")

def embed_query(q: str) -> list[float]:
    return openai_client.embeddings.create(
        model="text-embedding-3-small", input=[q]).data[0].embedding

def rag_answer(q: str) -> tuple[str, dict]:
    """Full RAG pipeline: retrieve → build prompt → LLM.
    Returns (answer, meta) where meta includes token counts."""
    top3 = dense_search(q, k=3)
    prompt, counts = build_rag_prompt(q, top3)
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.1,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = resp.choices[0].message.content
    meta = {
        "n_input_tokens":  resp.usage.prompt_tokens,
        "n_output_tokens": resp.usage.completion_tokens,
        "n_cached_input_tokens": getattr(resp.usage, 'prompt_tokens_details', None)
                                  and resp.usage.prompt_tokens_details.cached_tokens or 0,
        "n_embed_tokens":  len(q.split()),  # rough
    }
    return answer, meta

def cached_rag_answer(q: str, threshold: float = 0.95) -> tuple[str, dict]:
    """Cache-first RAG. Miss → full pipeline + store. Hit → return cached."""
    q_vec = embed_query(q)
    hit = cache_lookup(cache, q_vec, threshold=threshold)
    if hit:
        return hit["answer"], {
            "cache_hit": True,
            "similarity": hit["similarity"],
            "matched": hit["matched_question"],
        }
    answer, meta = rag_answer(q)
    cache_put(cache, q, q_vec, answer)
    meta["cache_hit"] = False
    return answer, meta

print("✓ Cache-first RAG function ready")
print("\nSanity check — first call MISSES (empty cache), second call HITS:")

q = "what is error code AC-1042"
print(f"\n  Q: {q!r}")

t0 = time.time()
ans1, meta1 = cached_rag_answer(q)
print(f"  Call 1: hit={meta1['cache_hit']}, {(time.time()-t0)*1000:.0f}ms → {ans1[:80]}...")

t0 = time.time()
ans2, meta2 = cached_rag_answer(q)
print(f"  Call 2: hit={meta2['cache_hit']}, sim={meta2.get('similarity'):.4f}, {(time.time()-t0)*1000:.0f}ms")

**Notice the latency difference on Call 2.** The cache hit skips the
retrieval AND the LLM call — you're only paying for one embedding of the
question. That's the ~800ms → ~15ms drop the deck (slide 19) advertises.

---

## Cell 6 — Threshold tuning: 3 paraphrases + a live false-hit

**The threshold controls the trade-off:**
- Too LOW (0.85): high hit rate, but false hits — you return the cached
  answer to a DIFFERENT question. Silent quality regression.
- Too HIGH (0.98): near-zero hit rate. Cache is useless.
- **Sweet spot (0.95):** hits on genuine paraphrases, misses on distinct
  questions. Programme default.

**We'll demo both sides live:**
1. **True positives** — three paraphrases of "what is AC-1042?" — cache
   should hit all three at 0.95
2. **False positive** — a superficially similar but distinct question
   about AC-4408 — cache MUST NOT hit at 0.95 (but does at 0.85)

In [ ]:
# ── Part 1: three paraphrases at three thresholds ──
# Cache is already primed with "what is error code AC-1042" from Cell 5.

paraphrases = [
    "what is error code AC-1042",       # original (from Cell 5) — expect exact match
    "AC-1042 error meaning",             # shorter paraphrase
    "tell me about the AC-1042 error",   # different phrasing
    "what does AC-1042 mean",            # another rephrasing
]
thresholds = [0.85, 0.95, 0.98]

print(f"══ Part 1: paraphrase hit rate at 3 thresholds ══\n")
print(f"  {'Query':<45s} | {'sim':>6s} | " + " | ".join(f"t={t}" for t in thresholds))
print(f"  {'-'*45} + {'-'*6} + " + " + ".join("-----" for _ in thresholds))

for p in paraphrases:
    p_vec = embed_query(p)
    # Get the raw similarity independent of threshold
    hit_any = cache_lookup(cache, p_vec, threshold=-1.0)  # -1.0 = always return match
    sim = hit_any["similarity"] if hit_any else 0.0
    
    hits_at = []
    for t in thresholds:
        h = "HIT" if sim >= t else "miss"
        hits_at.append(h)
    print(f"  {p[:45]:<45s} | {sim:>6.4f} | " + " | ".join(f"{h:>5s}" for h in hits_at))

print("\nExpected: all paraphrases hit at 0.85 and 0.95; some miss at 0.98")

In [ ]:
# ── Part 2: the FALSE HIT — this is the danger of low thresholds ──
# Cache has "what is error code AC-1042" (a query timeout).
# User asks about a DIFFERENT but SIMILAR-SOUNDING error, AC-4408 (dashboard failure).

distinct_q = "what is error code AC-4408"

q_vec = embed_query(distinct_q)
hit_at_min = cache_lookup(cache, q_vec, threshold=-1.0)
sim = hit_at_min["similarity"] if hit_at_min else 0.0

print(f"══ Part 2: false-hit demonstration ══\n")
print(f"  New question: {distinct_q!r}")
print(f"  Cached question that matches best: {hit_at_min['matched_question']!r}")
print(f"  Cosine similarity: {sim:.4f}\n")

for t in [0.85, 0.90, 0.95]:
    triggered = sim >= t
    verdict = 'CACHE HIT — returns WRONG answer (about AC-1042)' if triggered else 'cache miss — falls through to RAG (correct)'
    warning = ' ⚠' if triggered else ''
    print(f"  Threshold {t}: {verdict}{warning}")

# What's the actual cached answer we'd wrongly return at t=0.85?
print(f"\n  If we served the cached answer for {distinct_q!r} at t=0.85:")
print(f"  → {hit_at_min['answer'][:120]}...")
print(f"\n  User would get a plausible-looking answer to the WRONG question.")
print(f"  Silent quality regression. This is why deck slide 12 says")
print(f"  \"never ship < 0.95 without manual review of 20+ cache hits.\"")

**Discussion moment (this is the peak of Day 1):**
- What's the false-hit rate you can tolerate? (For most enterprise use, ≤1%.
  Every false hit is a plausible-but-wrong answer served with confidence —
  and the user won't know.)
- Would you rather MISS a cache opportunity (pay full RAG cost) or serve
  a wrong answer? (Always miss. Always.)
- What would you monitor in production? (False-hit rate via periodic
  sampling — pull N cache hits per day, replay through full RAG, check
  answer agreement.)

---

## Cell 7 — Layer 2: OpenAI prompt caching (provider-side)

**Different mechanism from Layer 1.** The semantic cache we just built
stores question→answer pairs on OUR side. The provider prompt cache stores
the KV state of your PREFIX on OPENAI'S side.

**Automatic on OpenAI when your prompt ≥ 1,024 tokens.** No code change.
Explicit control available on GPT-5.6+ (out of scope this cohort).

**How to see it fire:** OpenAI's response returns `prompt_tokens_details.
cached_tokens`. Non-zero = the cache fired. Bill the cached portion at
the discounted rate (50-90% off depending on model — see deck v2 slide 13).

**Prompt-structure rule:** put stable content (system prompt, few-shot)
FIRST. User question + retrieved chunks LAST. If you swap the order, the
prefix keeps changing every call and the cache never fires.

In [ ]:
# Run the SAME prompt structure twice, inspect cached_tokens field.
# We use a longer question to push the prompt past 1024 tokens.
test_q = ("I need a detailed explanation of what causes error AC-1042 in the "
          "Acme Analytics Platform, what the resolution steps are, whether it "
          "relates to Snowflake connections, and what the default timeout is.")

print(f"══ Prompt cache demonstration ══\n")

# Bypass the semantic cache — call RAG directly twice
for run in [1, 2]:
    t0 = time.time()
    _, meta = rag_answer(test_q)
    elapsed_ms = (time.time() - t0) * 1000
    cached = meta['n_cached_input_tokens']
    total_input = meta['n_input_tokens']
    pct_cached = 100 * cached / total_input if total_input else 0
    print(f"  Run {run}: input={total_input:4d} tokens, cached={cached:4d} ({pct_cached:.0f}%), {elapsed_ms:.0f}ms")

print(f"\nExpected pattern:")
print(f"  Run 1: cached=0 (cold — nothing in provider cache yet)")
print(f"  Run 2: cached=~950 (prefix matched; discounted billing)")
print(f"\nIf Run 2 shows cached=0 too, either:")
print(f"  • Prompt is <1024 tokens (below OpenAI's threshold)")
print(f"  • Prompt prefix differs somehow (check that system prompt is byte-identical)")
print(f"  • Model doesn't support prompt cache (gpt-4o-mini does; verify current docs)")

---

## Cell 8 — Layer 3: embedding cache (30-second acknowledgement)

**You already have this from W7.** When you `embed_texts(["hello", "world"])`
the same list twice, the second call should be free (in-process memoization
via `@lru_cache` or a persistent cache on disk).

Embedding cache prevents re-embedding the same text repeatedly during
ingestion of a document with many chunks or during query rewrites (HyDE,
multi-query — each may embed the same text more than once).

**Nothing new to build today.** Just remember it's Layer 3.

Three layers summary:
- **Layer 1 (Semantic cache)** — us storing question→answer. Biggest win.
- **Layer 2 (Prompt cache)** — provider caches KV state of prefix. Free with OpenAI.
- **Layer 3 (Embedding cache)** — us memoising embeddings. Already exists from W7.

---

## Cell 9 — Cost + latency: cold vs warm across 6 queries

Time to measure the promised 60-70% savings. Run all 6 test queries twice:

- **Cold pass:** empty semantic cache; every query MISSES → full RAG runs
- **Warm pass:** same queries again → every query HITS → cache returns answer

**What we expect (from deck v2 slide 19-20):** cost drops 60-70% ON THE
WARM PASS (the deck v2 wording — the auditor rightly flagged the original
'60-70%' as unconditional). Latency p50 drops dramatically (LLM call
eliminated on hits).

In [ ]:
# Clear the cache so we're truly cold
cache_clear(cache)
print(f"✓ Cache cleared for cold pass\n")

def run_pass(label):
    total_cost, latencies = 0.0, []
    n_hits, n_misses = 0, 0
    for tq in queries:
        t0 = time.time()
        ans, meta = cached_rag_answer(tq["q"])
        elapsed_ms = (time.time() - t0) * 1000
        latencies.append(elapsed_ms)
        if meta["cache_hit"]:
            n_hits += 1
            # Cache hit cost = only the query-embedding call
            total_cost += estimate_cost_usd(0, 0, 0, n_embed_tokens=count_tokens(tq["q"]))
        else:
            n_misses += 1
            total_cost += estimate_cost_usd(
                n_input_tokens=meta["n_input_tokens"],
                n_output_tokens=meta["n_output_tokens"],
                n_cached_input_tokens=meta["n_cached_input_tokens"],
                n_embed_tokens=count_tokens(tq["q"]),
            )
    p50 = sorted(latencies)[len(latencies)//2]
    return {"label": label, "cost": total_cost, "p50_ms": p50,
            "hits": n_hits, "misses": n_misses, "total": len(queries)}

cold = run_pass("cold")
warm = run_pass("warm")

print(f"══ Cold vs Warm across {len(queries)} queries ══\n")
print(f"  {'Pass':<8s}  {'hits':>5s}  {'misses':>7s}  {'p50 ms':>8s}  {'cost':>10s}")
print(f"  {'----':<8s}  {'----':>5s}  {'------':>7s}  {'------':>8s}  {'----':>10s}")
print(f"  {cold['label']:<8s}  {cold['hits']:>5d}  {cold['misses']:>7d}  {cold['p50_ms']:>8.0f}  ${cold['cost']:>9.5f}")
print(f"  {warm['label']:<8s}  {warm['hits']:>5d}  {warm['misses']:>7d}  {warm['p50_ms']:>8.0f}  ${warm['cost']:>9.5f}")

if cold['cost'] > 0:
    cost_saved_pct = 100 * (cold['cost'] - warm['cost']) / cold['cost']
    latency_saved_pct = 100 * (cold['p50_ms'] - warm['p50_ms']) / cold['p50_ms']
    print(f"\n  Warm-pass cost saved:    {cost_saved_pct:.1f}%")
    print(f"  Warm-pass latency saved: {latency_saved_pct:.1f}%")

**On repeat-query workloads** (support tickets, FAQ-style traffic, docs
search), you should see 60-90% savings on the warm pass. On highly
unique-query workloads (novel research, exploratory analytics), the
semantic cache hit rate is much lower and total savings shrink.

**Real production numbers land somewhere between the cold pass (0% hits)
and the warm pass (100% hits). Measure yours before making cost claims.**

---

## Cell 10 — Wrap + Day 2 preview

**Today (Day 1) you built:**
1. Prompt token accounting (know where your tokens live before cutting)
2. Reduce-k with precision AND recall check (not precision alone)
3. SQLite-backed semantic cache (~120 lines total)
4. Live threshold tuning + false-hit demo (why 0.95 is the default)
5. Prompt cache inspection (`cached_tokens` field on OpenAI response)
6. Cost + latency comparison (cold vs warm)

**Tomorrow (Day 2) covers:**
- Document update flow (the naive duplicate-insertion problem)
- Tombstone pattern (soft delete via `deleted_at` payload)
- **A live IsNull vs IsEmpty bug diagnosis** — the tombstone filter bug
  the auditor caught. We'll write it wrong first, watch it silently fail,
  fix with `IsEmptyCondition`, understand why
- **Cache invalidation gotcha** — the semantic cache we built today serves
  stale answers after a document update. Fixing that is a first-class step
  in the update runbook
- Re-indexing (with the corrected cost formula from slide 31)
- KPI comparison across W8→W9→W10

**Track B (take-home):** wire the cache into YOUR capstone's
`qdrant_rag.py`, run golden set cold + warm, update wk10-snapshot.md.
Core/Stretch tagging continues.

---

## Homework before Day 2 (~2 min)

**No downloads this week** (unlike W9's cross-encoder). Just verify:

```python
from qdrant_client.models import IsEmptyCondition, IsNullCondition
# both should import without error — if IsEmpty fails, your qdrant-client
# is very old (pre-1.7). Upgrade with: pip install -U qdrant-client
```

That's it. See you tomorrow for the tombstone bug hunt.